# Tutorial 11: Key-Based Authentication and Authorization (Offline-First)

Focused tutorial for `A2A/11_authentication_authorizatoin_key_based`, centered on API-key protected invocation in A2A/MCP calls.

## 1) Where We Are in the Journey

`10_Human_in_loop` added governance through approvals.
This step adds service-level trust checks with API keys.
It exists to prevent unauthorized invocation of agent endpoints.

## 2) What You Will Learn

- Header-based API key verification
- Where authorization checks sit in invocation path
- Failure responses for missing/invalid keys
- How key-based auth fits A2A infrastructure

## 3) Why This Matters

Without authentication, any caller can invoke tools.
Key-based checks provide a simple first security boundary.
This is foundational before moving to token-based auth.

## 4) Core Concept Deep Dive

Folder test cell sends `headers={"x-api-key": "controller-key"}` with invoke payload.
Security model: verify caller key before tool execution.
Trade-off: easy setup but weaker than signed/expiring tokens.

## 5) Code Walkthrough (Only `A2A/11_authentication_authorizatoin_key_based`)

- `test.ipynb` posts to `/invoke` with tool args and callback URL.
- Request includes `x-api-key` header.
- Tutorial mirrors that exact shape and demonstrates allow/deny behavior offline.

In [ ]:
VALID_KEYS = {'controller-key'}

def protected_invoke(payload, headers):
    api_key = headers.get('x-api-key')
    if api_key not in VALID_KEYS:
        return {'status': 'error', 'code': 401, 'message': 'Unauthorized'}

    if payload.get('tool') == 'add':
        a = payload['args']['a']
        b = payload['args']['b']
        return {'status': 'success', 'result': a + b}
    return {'status': 'error', 'code': 400, 'message': 'Unknown tool'}

In [ ]:
payload = {'tool': 'add', 'args': {'a': 10, 'b': 20}, 'callback_url': 'http://localhost:9000/callback'}
print('VALID KEY:', protected_invoke(payload, {'x-api-key': 'controller-key'}))
print('INVALID KEY:', protected_invoke(payload, {'x-api-key': 'bad-key'}))
print('MISSING KEY:', protected_invoke(payload, {}))

## 6) System Flow Explanation

1. Client prepares invoke payload.
2. Client includes API key header.
3. Service verifies key before execution.
4. If authorized, tool executes.
5. If unauthorized, request is rejected.

## 7) Important Concepts You Might Miss

- Auth check should happen before parsing expensive logic.
- Key scope/rotation policy is part of system design.
- Security events should be logged for traceability.

## 8) Common Mistakes / Pitfalls

- Hardcoding keys without rotation strategy.
- Returning success-like responses on auth failure.
- Forgetting to protect callback-triggering endpoints.

## 9) Key Takeaways

- Key-based auth is a practical first security layer.
- Header validation must gate every protected call.
- This step prepares transition to JWT-based auth.

## 10) What’s Next (Strict Preview)

`12_JWT_authentication_authorizatoin` introduces token-based auth with richer identity and expiry semantics.
It solves limitations of static API keys and enables stronger authorization controls.
This matters for scalable multi-tenant agent platforms.